# Saving and Publishing 

Now that you've run the workflow in the previous notebook, it's time to turn that data into something familiar that you can use. In this section, we'll turn the output JSON file into a spreadsheet and a static website.  


[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1graycCkEiGLwMlFIoZg3-eDAXH5cKYkP?usp=sharing)  

In [ ]:
from pathlib import Path

output_path = "output"

data_files = Path(output_path).glob("*.json")
data_files = list(data_files)
print(f"Found {len(data_files)} file(s)")

In [ ]:
# name, image_uri, cleaned_text, box_metadata>"Idenfifier"
import srsly
from csv import DictWriter

def output_json_to_spreadsheet(data_files: list):
  rows = []
  for data_file in data_files:
    data = srsly.read_json(data_file)
    box_identifier = data.get('box_metadata', 'unlabeled_box').get('Identifier','')
    images = data.get('images',[])
    # sort the images by filename
    images = sorted(images, key=lambda x: x.get('name',''))
    for image in images:
      rows.append({
          'name': image.get('name',''),
          'image_uri': image.get('image_uri',''),
          'cleaned_text': image.get('cleaned_text',''),
          'box_identifier': box_identifier
      })
  writer = DictWriter(open('output.csv','w'), fieldnames=rows[0].keys())
  writer.writeheader()
  writer.writerows(rows)

output_json_to_spreadsheet(data_files)

## Create a static website

In [ ]:
import srsly
import markdown
from pathlib import Path

if not Path('_site').exists():
  Path('_site').mkdir()

for data_file in data_files:
    data = srsly.read_json(data_file)
    images = data.get('images',[])
    images = sorted(images, key=lambda x: x.get('name',''))
    for idx, image in enumerate(images):
        next_image = images[idx+1]['name'] + ".html" if idx < len(images)-1 else None
        prev_image = images[idx-1]['name'] + ".html" if idx > 0 else None
        cleaned_text = image.get('cleaned_text','')
        cleaned_text = markdown.markdown(cleaned_text, extensions=['tables'])
        sort_value = int("".join([char for char in image["name"] if char.isdigit()]))
        html_template = f"""
        <!DOCTYPE html>
        <html>
          <body>
            <button><a href="/">Back to Search</a></button>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
            <button><a href="{prev_image}">Back</a></button>
            <button><a href="{next_image}">Forward</a></button>
          
            <h1 data-pagefind-sort="page:{sort_value}">{image["name"]}</h1>
            <img src="{image["image_uri"]}">
            <p>{cleaned_text}</p>
            <table>
              <tr>
                <th>Entity</th>
                <th>Type</th>
                <th>Context</th>
              </tr>
              {[f"<tr><td>{entity["entity"]}</td><td>{entity["type"]}</td><td>{entity["context"]}</td></tr>" for entity in image["entities"] if isinstance(entity, dict) and entity.get('entity',None) and entity.get('type',None) and entity.get('context', None)]}
            </table>
          </body>
        </html>
        """
        Path(f"_site/{image['name']}.html").write_text(html_template)

In [ ]:
full_metadata_html = ""
for data_file in data_files:
    data = srsly.read_json(data_file)
    box_metadata = data.get('box_metadata',{})
    identifier = box_metadata.get('Identifier','no_identifier')
    full_metadata_html += f"<h1>{identifier}</h1>"
    for key, value in box_metadata.items():
      full_metadata_html += f"<h2>{key}</h2>"
      full_metadata_html += f"<p>{value}</p><br>"
    full_metadata_html += "<hr>"

index = f"""

<!DOCTYPE html>
        <html>
          <head>
          <link href="/pagefind/pagefind-ui.css" rel="stylesheet">
          <script src="/pagefind/pagefind-ui.js"></script>
          </head>
          <body>
            <div id="search"></div>
            {full_metadata_html}"""
index += """
            <script>
                window.addEventListener('DOMContentLoaded', (event) => {
                    new PagefindUI({
                      element: "#search",
                      sort: { date: "desc" },
                      showSubResults: true
                    });

                });
            </script>
             </body>
        </html>
"""
Path('_site/index.html').write_text(index)

In [ ]:
%pip install 'pagefind[extended]'

In [ ]:
!python3 -m pagefind --site _site

In [ ]:
import shutil
zip_file = shutil.make_archive('site', 'zip', '_site')

Netlify other options, deploy to the web

https://eap617-1-10.netlify.app/

While it's great to have a spreadsheet or static page to work from, ideally we'll find ways for transcriptions and data to find their way back to the British Library and other libraries and museums. Librarians and archivists are hard at work developing this capability.  For example, the Swedish National Archives, recently added the ability to search over 3 million hand-written documents.  

- [Here is a post about this work (in Swedish and English)](https://riksarkivet.se/inlagg/riksarkivet-gor-en-miljon-handskrivna-dokument-sokbara)
- [Here is an example search result](https://sok.riksarkivet.se/bildvisning/A0065892_00072#?q=blau&xywh=-957%2C0%2C7736%2C4863&hi=3&cv=71) 